# Visualization of a linear model fit to a dataset of house prices.

First, we will read the data and create a scatter plot to observe them.

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt

url = 'https://raw.githubusercontent.com/noe-tec/machine-learning/main/datasets/example_data_price_houses_v2.csv'
response = requests.get(url)

if response.status_code == 200:
    lines = response.text.strip().splitlines()
    data = [list(map(float, line.split(','))) for line in lines]
    for row in data[:5]:  # show the first 5 rows
        print(row)
    data = np.array(data)
    print(data.shape)
    
    x_train = data[:,  0] 
    y_train = data[:, -1]

    plt.scatter(x_train, y_train)
    plt.xlabel("Area [m^2]")
    plt.ylabel("Price [millons of mxn]")
    plt.ylim((0,9))
    plt.grid()
else:
    print(f'Error downloading the file: {response.status_code}')




Now we are going to create an interactive plot that allows us to select values for the parameters "w" and "b" and see the resulting line. The plot also shows a contour plot of the cost function J(w, b) and the current value of the cost for the chosen parameters.

In [ ]:
from ipywidgets import interact, FloatSlider

# --- Helper functions ---
def prediction(x, w, b):
    return w * x + b

def calculate_cost(x, y, w, b):
    predictions = prediction(x, w, b)
    squared_errors = (predictions - y)**2
    mean_squared_error = squared_errors.mean()
    return mean_squared_error

# --- We create a grid with combinations of w and b and calculate the cost for each combination ---
w_range = np.linspace(-0.2, 0.2, 100)
b_range = np.linspace(-6, 6, 100)
w_values, b_values = np.meshgrid(w_range, b_range)
cost_values = np.zeros_like(w_values)

for i in range(w_values.shape[0]):
    for j in range(w_values.shape[1]):
        cost_values[i, j] = calculate_cost(x_train, y_train, w_values[i, j], b_values[i, j])

# We create a function to visualize the fitted line and the cost
def plot_visualization(w, b):
    fig, axs = plt.subplots(1, 2, figsize=(16, 6))

    # Left subplot: regression
    axs[0].scatter(x_train, y_train, label='Actual data')
    x_line = np.linspace(x_train.min(), x_train.max(), 10)
    y_line = prediction(x_line, w, b)
    axs[0].plot(x_line, y_line, color='red', label=f'y = {w:.3f}x + {b:.3f}')
    axs[0].set_title(f'Linear regression\nCost (MSE): {calculate_cost(x_train, y_train, w, b):,.2f}')
    axs[0].set_xlabel("Size [m²]")
    axs[0].set_ylabel("Price [mxn]")
    axs[0].set_ylim((-10, 10))
    axs[0].set_xlim((0, x_train.max()))
    axs[0].grid(True)
    axs[0].legend()

    # Right subplot: contours of the cost function
    levels = np.logspace(0, 2.5, 10)  # We use logarithmic levels for better visualization
    CS = axs[1].contour(w_values, b_values, cost_values, levels=levels, cmap='jet')
    axs[1].set_title("Contours of the cost function")
    axs[1].set_xlabel("w (slope)")
    axs[1].set_ylabel("b (intercept)")
    axs[1].plot(w, b, 'ro', label='Current position')
    axs[1].legend()
    axs[1].grid(True)

    plt.tight_layout()
    plt.show()

# Interactive graph
interact(plot_visualization,
         w=FloatSlider(min=-0.1, max=0.1, step=0.001, value=0, description='w (slope)'),
         b=FloatSlider(min=-3, max=3, step=0.01, value=0, description='b (intercept)'))

Actividad: Modifica los valores de "w" y "b" usando los sliders. Observa como la linea resutante varía y el costo cambia. ¿Que valores de w y b producen un buen ajuste?